# Week 6 · 03 Training stability: initialization, normalization, residual connections

`01` §5 showed the problem: in a deep network the signal going forward, and the gradient going back, are multiplied once per layer, so they can shrink to nothing or blow up. This notebook covers the four standard tools that keep deep networks trainable: **initialization** scaled to the layer size, **batch normalization** (and its cousin layer normalization), **residual connections**, and **gradient clipping**. Each one gets the problem, the idea, the formula piece by piece, a small example, and its PyTorch line. Reading: UDL Ch. 7.5 and 11; d2l.ai §5.4, §8.5 and §8.6.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **initialization**: the random values the weights start from, before any training.
- **variance, std**: how spread out a set of numbers is (std = the square root of the variance). Here: the spread of a layer's outputs.
- **normalization**: shifting and scaling numbers so that they have mean 0 and standard deviation 1.
- **running average**: an average kept up to date as new batches arrive (an EWMA, like Adam's).
- **residual (skip) connection**: adding a block's input to its output: out = x + F(x).

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from figures import *
import checks; checks.load(globals(), "03")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** Why this notebook exists: a 22-layer network on the digits. Plain, it never gets below the loss of random guessing (2.30). With batch normalization after every layer, the same network learns.

In [ ]:
fig_deep_training()

| Section | What | Why it's here |
|---|---|---|
| 0 | recap, and booklet 14's Adam from memory | the optimizer every model this week uses |
| E | the four tools at a glance | start from the end: what each one fixes |
| 1 | initialization | start with weights of the right size: a proposition about variances |
| 2 | batch normalization | re-centre every layer's values on every batch |
| 3 | layer normalization | the same, per sample instead of per feature |
| 4 | residual connections | a path where the gradient is multiplied by 1 |
| 5 | gradient clipping | a cap for gradients that blow up |
| 6 | In Python | one block with all of them; exercises 2–5 |

## 0. Recap first · 10 min, closed book

**Quick check:** Why do gradients vanish in a deep sigmoid network?

<details><summary>Answer</summary>

Backprop multiplies one local gradient per layer, and $\sigma'(z) \le 0.25$, so after many layers the product is tiny. [01 §5 ↗](01_gd_and_backprop_slowly.ipynb)

</details>

**Quick check:** The variance of a sum of independent random numbers?

<details><summary>Answer</summary>

The sum of their variances: $\operatorname{Var}(a + b) = \operatorname{Var}(a) + \operatorname{Var}(b)$ for independent $a, b$. [Week 1 · 03 §2 ↗](../week1/03_math_essentials.ipynb)

</details>

**Quick check:** Standardizing a feature: the formula?

<details><summary>Answer</summary>

$z = \frac{x - \mu}{\sigma}$, with the mean and std of the training data. Your booklet, chapter 11. [Week 5 · 01 ex. 5 ↗](../week5/01_formula_to_numpy.ipynb)

</details>

**Quick check:** Adam's update in one line?

<details><summary>Answer</summary>

$w \leftarrow w - \eta\, \hat m / (\sqrt{\hat s} + \varepsilon)$, with bias-corrected averages of $g$ and $g^2$. [Week 5 · 05 §5 ↗](../week5/05_optimizers.ipynb)

</details>

**Quick check:** A residual block from Week 5 · 06 §10: what did it compute, and why must its input and output shapes match?

<details><summary>Answer</summary>

$x + F(x)$; the two are added entry by entry. [Week 5 · 06 §10 ↗](../week5/06_pytorch_basics.ipynb)

</details>

**Exercise 1 · Booklet 14: Adam.** Exactly your booklet's function: `st` is a dictionary the function keeps its memory in between calls (empty at the start). Each call: `st["t"]` goes up by 1; $m \leftarrow \beta_1 m + (1 - \beta_1) g$; $s \leftarrow \beta_2 s + (1 - \beta_2) g^2$; bias-correct both; return $w - \eta\, \hat m / (\sqrt{\hat s} + \varepsilon)$. Use `st.get("m", 0)` for "0 the first time".

**Example**: what goes in, and what should come out:
```python
st = {}
w = adam(np.array([1.0]), np.array([2.0]), st, 0.1)    # → [0.9]   the first step has size η
st["t"]                                                # → 1
```

In [ ]:
def adam(w, g, st, lr, b1=0.9, b2=0.999, eps=1e-8):
    pass  # your code

In [ ]:
check(1)

## E. Start from the end: the four tools at a glance

| Tool | The problem it fixes | What it does | In PyTorch |
|---|---|---|---|
| He / Xavier initialization | the signal shrinks or grows layer after layer from the very first step | starts each weight with variance $2 / n_{\text{in}}$ (ReLU) or $1 / n_{\text{in}}$ | `nn.init.kaiming_normal_(w)`; `nn.Linear`'s default is close |
| batch normalization | each layer's inputs drift in scale as the layers before it change | rescales each feature to mean 0, std 1 over the batch, then a learned scale and shift | `nn.BatchNorm1d(d)` / `nn.BatchNorm2d(c)` |
| layer normalization | the same, when batches are tiny or have no fixed shape (text) | rescales each sample over its own features | `nn.LayerNorm(d)` |
| residual connection | gradients vanish through many layers | adds the block's input to its output, so the gradient has a path multiplied by 1 | `return x + self.block(x)` |
| gradient clipping | one huge gradient wrecks the weights in one step | shrinks the gradient if it's longer than a limit | `nn.utils.clip_grad_norm_(params, 1.0)` |

In a modern network you'll see several together: ResNets (`05`) use He initialization, batch normalization and residual connections in every block; Transformers (Week 8) use layer normalization, residual connections, and often clipping.

## 1. Initialization: start with weights of the right size

**The problem.** Before training, the weights are random. If they're too small, each layer's output is smaller than its input, and after 12 layers the signal is practically 0; too large, and it explodes. Either way the gradients do the same thing (`01` §5), so training can't get going.

🖼 **Picture.** How spread out each layer's outputs are at the start, for three choices of the weights' standard deviation. Too small: the spread falls by about 10 times per layer. Too large: it grows. He's value keeps it steady.

In [ ]:
fig_init_spread()

**The idea.** Choose the size of the random starting weights so that each layer's output has the **same spread** as its input. Then a signal can pass through any number of layers unchanged in size.

**Proposition.** Let $z = \sum_{j=1}^{n} w_j x_j$ (one unit with $n$ inputs), where all the $w_j$ and $x_j$ are independent, have mean 0, $\operatorname{Var}(w_j) = v_w$ and $\operatorname{Var}(x_j) = v_x$. Then $\operatorname{Var}(z) = n\, v_w\, v_x$.

**What it says, in words.** A unit adds up $n$ random products, and adding up $n$ independent things adds up their variances. So the output's spread is $n$ times the spread of one product. To keep $\operatorname{Var}(z) = v_x$ (same as the input), choose $v_w = 1/n$.

**A small example.** $n = 256$ inputs, each with variance 1. With weights of variance $1/256$ (std $0.0625$): $\operatorname{Var}(z) = 256 \times \frac{1}{256} \times 1 = 1$. With std 0.2 (variance 0.04): $\operatorname{Var}(z) = 256 \times 0.04 = 10.24$, so the std grows by $\sqrt{10.24} = 3.2$ times **per layer**, which is the orange line in the picture.

**The proof in plain words.** Each product $w_j x_j$ has mean 0 (independent, both mean 0) and variance $\mathbb{E}[w_j^2 x_j^2] = \mathbb{E}[w_j^2]\, \mathbb{E}[x_j^2] = v_w v_x$. The products are independent, so the variance of their sum is the sum of their variances:

$$
\operatorname{Var}(z) = \sum_{j=1}^{n} \operatorname{Var}(w_j x_j) = \sum_{j=1}^{n} v_w\, v_x = n\, v_w\, v_x. \qquad \blacksquare
$$

**Remark 1: ReLU needs twice as much (He initialization).** ReLU sets the negative half of its inputs to 0, which halves the average of the squares that the next layer receives. To make up for it, double the weight variance: $v_w = 2 / n_{\text{in}}$, std $= \sqrt{2 / n_{\text{in}}}$. That's **He** (or Kaiming) initialization, for ReLU networks. For tanh or sigmoid, **Xavier** (Glorot) uses about $1 / n_{\text{in}}$ (precisely $2 / (n_{\text{in}} + n_{\text{out}})$, which also balances the backward pass).

**Remark 2: why random at all?** If every weight started at the same value, every unit in a layer would compute the same thing and get the same gradient, and they'd stay identical forever (Week 5 · 02 §10). Random starting values break that symmetry; the proposition only says how large they should be.

**Remark 3: in PyTorch.** `nn.Linear` starts its weights uniformly in $\pm 1/\sqrt{n_{\text{in}}}$, a variance of $\frac{1}{3 n_{\text{in}}}$, a bit smaller than He. For deep ReLU networks, set He explicitly with `nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")`.

In [ ]:
rng = np.random.default_rng(0)
n = 256
x = rng.normal(0, 1, size=(10000, n))                  # 10,000 inputs, each of the n features with variance 1
W = rng.normal(0, np.sqrt(1 / n), size=(n, 64))        # weights with variance 1/n
print((x @ W).var().round(3))                          # → 0.981   about 1 = n · (1/n) · 1

**In one sentence:** a unit's output variance is $n_{\text{in}}$ times its weights' variance times its inputs' variance, so starting the weights with variance $1/n_{\text{in}}$ (or $2/n_{\text{in}}$ for ReLU) keeps signals the same size through every layer.

## 2. Batch normalization

**The problem.** Good initialization only fixes the start. During training, every layer's weights change, so the values flowing into the next layer keep changing in scale and centre. Each layer is chasing a moving target, and deep networks train slowly or not at all (the plain line in §M).

**The idea.** After a layer's weighted sum, **standardize each feature over the batch**: subtract the batch's mean of that feature and divide by its std, exactly like feature scaling (your booklet, chapter 11), but done inside the network, on every batch. Then let the network learn, for each feature, the scale and shift it actually wants.

🖼 **Picture.** A batch of 4 samples × 3 features. Batch norm works **down each column** (one feature across the batch). Layer norm (§3) works **along each row** (one sample across its features).

In [ ]:
fig_bn_vs_ln()

**The formula, piece by piece**, for a batch $X$ of shape $(n, d)$ (rows = samples, columns = features):

$$
\mu_j = \frac{1}{n}\sum_{i=1}^{n} x_{ij}, \qquad \sigma_j^2 = \frac{1}{n}\sum_{i=1}^{n}(x_{ij} - \mu_j)^2, \qquad \hat x_{ij} = \frac{x_{ij} - \mu_j}{\sqrt{\sigma_j^2 + \varepsilon}}, \qquad y_{ij} = \gamma_j\, \hat x_{ij} + \beta_j
$$

- $\mu_j$, $\sigma_j^2$: the mean and variance of **column** $j$ over the $n$ samples of this batch.
- $\hat x_{ij}$: column $j$ standardized. $\varepsilon$ (like $10^{-5}$) only stops a division by 0.
- $\gamma_j$ (scale) and $\beta_j$ (shift) are **learned parameters**, one pair per feature, starting at 1 and 0. They let the network undo the normalization if it wants: $\gamma_j = \sigma_j$, $\beta_j = \mu_j$ gives back $x$.
- In NumPy: `X.mean(axis=0)` and `X.var(axis=0)`: axis 0, because the rows (samples) are what's averaged away.

**A small example** (the picture's batch): column 1 holds $50, 70, 60, 40$. Mean $\mu_1 = 55$; variance $\frac{25 + 225 + 25 + 225}{4} = 125$; std $11.18$. So $50 \to \frac{50 - 55}{11.18} = -0.447$ and $70 \to 1.342$. After this, every column has mean 0 and std 1, however large its raw values were.

In [ ]:
X = np.array([[1.0, 50.0, 0.2], [3.0, 70.0, 0.4], [5.0, 60.0, 0.1], [7.0, 40.0, 0.3]])   # (4, 3): 4 samples, 3 features
by_hand = (X - X.mean(axis=0)) / np.sqrt(X.var(axis=0) + 1e-5)      # gamma = 1, beta = 0 at the start
bn = nn.BatchNorm1d(3)                                               # 3 = the number of features
print(by_hand.round(3))
print(np.allclose(by_hand, bn(torch.tensor(X, dtype=torch.float32)).detach().numpy(), atol=1e-4))
# → [[-1.342 -0.447 -0.447]
#    [-0.447  1.342  1.341]
#    [ 0.447  0.447 -1.341]
#    [ 1.342 -1.342  0.447]]
#   True

**Training vs testing: the running averages.** At test time there may be a single image, and its prediction mustn't depend on which other images happen to be in the batch. So while training, batch norm also keeps **running averages** of each feature's mean and variance (an EWMA: 90% old + 10% this batch, by default). `model.eval()` makes it use those stored values instead of the batch's. After just **one** batch the running mean is only $0.1 \times 55 = 5.5$ for feature 1 (it started at 0), which is why evaluation mode gives quite different numbers until many batches have been seen:

In [ ]:
print(bn.running_mean, bn.running_var)       # after the one batch above (PyTorch's running variance divides by n - 1)
bn.eval()                                   # testing: use the running averages, not the batch
print(bn(torch.tensor(X[:1], dtype=torch.float32)).detach().numpy().round(3))   # one sample alone is fine now
# → tensor([0.4000, 5.5000, 0.0250]) tensor([ 1.5667, 17.5667,  0.9017])
#   [[ 0.479 10.617  0.184]]

**Remarks.** (1) It's why `model.train()` and `model.eval()` matter (your "two switches"). (2) Put it between the weighted sum and the activation: `Linear → BatchNorm1d → ReLU`; for images `Conv2d → BatchNorm2d → ReLU`, normalizing each channel over the batch and all pixel positions. (3) Batch sizes below about 8 make $\mu_j$ and $\sigma_j$ unreliable; that's when layer norm is used instead. (4) A side effect: the layer before no longer needs its own bias, since $\beta_j$ does that job (`nn.Linear(..., bias=False)` is common before batch norm).

**In one sentence:** batch norm standardizes each feature over the batch and then applies a learned scale and shift, using stored running averages at test time.

## 3. Layer normalization

**The problem.** Batch norm needs a decent batch: its statistics come from the other samples. With a batch of 1 or 2, or in models that process text one word at a time (Week 8), there's nothing reliable to average over.

**The idea.** Do the same standardization, but **along each sample's own features** (each row) instead of down each feature (each column). Every sample is normalized on its own, so the batch size doesn't matter, and training and testing behave identically (no running averages).

$$
\mu_i = \frac{1}{d}\sum_{j=1}^{d} x_{ij}, \qquad \sigma_i^2 = \frac{1}{d}\sum_{j=1}^{d}(x_{ij} - \mu_i)^2, \qquad y_{ij} = \gamma_j\, \frac{x_{ij} - \mu_i}{\sqrt{\sigma_i^2 + \varepsilon}} + \beta_j
$$

The only change from §2 is the index: the mean and variance are now over $j$ (features) for each sample $i$. In NumPy: `axis=1, keepdims=True` (one number per row, kept as a column so it lines up with the rows). Sample 2 of the batch, $(5, 60, 0.1)$, has mean $21.7$, so its values become $(-0.615, 1.410, -0.795)$ whatever the other samples are.

In [ ]:
by_hand = (X - X.mean(axis=1, keepdims=True)) / np.sqrt(X.var(axis=1, keepdims=True) + 1e-5)
ln = nn.LayerNorm(3)                                                 # 3 = the size of each sample's feature vector
print(by_hand[2].round(3), np.allclose(by_hand, ln(torch.tensor(X, dtype=torch.float32)).detach().numpy(), atol=1e-4))
# → [-0.615  1.41  -0.795] True

**In one sentence:** layer norm is batch norm's formula applied to each sample's own features instead of each feature across the batch, so it works with any batch size and needs no train/test switch.

## 4. Residual connections

**The problem.** Even with good initialization and normalization, very deep stacks (50, 100 layers) were found to train **worse** than shallower ones, because the gradient still passes through a long product of factors on its way back.

**The idea.** Give each block a shortcut: add its input straight to its output, $\text{out} = x + F(x)$. The block's layers $F$ then only have to learn a **correction** to $x$ (if nothing needs changing, $F$ can just output 0). And the gradient gets a path that skips the layers entirely.

🖼 **Picture.** A residual block: two layers $F$ plus the skip connection (orange) that carries $x$ around them to the $+$.

In [ ]:
fig_residual_block()

**Proposition** (one number per layer, to keep it simple). If $\text{out} = x + F(x)$, then $\frac{d\,\text{out}}{dx} = 1 + F'(x)$. Through $L$ such blocks, the gradient is multiplied by $\prod_{l=1}^{L}(1 + F_l')$ instead of $\prod_{l=1}^{L} F_l'$.

**What it says, in words.** In a plain network each layer multiplies the gradient by its own small factor $F_l'$. With the skip connection, each block multiplies it by $1 + F_l'$, which stays close to 1 when $F_l'$ is small. The gradient can't fade away just because the layers are many.

**A small example.** Ten blocks whose layers each have $F' = 0.1$. Plain: $0.1^{10} = 10^{-10}$, the gradient is gone. Residual: $1.1^{10} = 2.59$, perfectly usable.

**The proof in plain words.** The derivative of a sum is the sum of the derivatives, and $\frac{dx}{dx} = 1$: $\frac{d}{dx}(x + F(x)) = 1 + F'(x)$. Chaining $L$ blocks multiplies their derivatives (the chain rule), giving the product. $\blacksquare$ (With vectors, the 1 becomes the identity matrix: $I + \partial F / \partial x$.)

**Remarks.** (1) The input and output of $F$ must have the same shape, since they're added; when a block changes the size, the skip path gets its own small layer to match (`05` §5's ResNet code). (2) This one idea made 100+ layer networks trainable (ResNet, 2015), and every Transformer uses it.

In [ ]:
for residual in [False, True]:                       # 20 sigmoid layers; the gradient at the FIRST layer
    torch.manual_seed(0)
    layers = [nn.Linear(32, 32) for _ in range(20)]
    h = torch.randn(64, 32)
    for lin in layers:
        f = torch.sigmoid(lin(h))
        h = h + f if residual else f                 # the only difference: add the input back
    h.sum().backward()
    print("residual" if residual else "plain   ", f"{layers[0].weight.grad.norm().item():.1e}")
# → plain    2.5e-15
#   residual 5.3e+01

**In one sentence:** a residual connection adds a block's input to its output, so the gradient is multiplied by $1 + F'$ per block instead of $F'$, and deep networks stay trainable.

## 5. Gradient clipping

**The problem.** Now and then a batch produces a huge gradient (deep networks, recurrent networks, a bad batch). One step with it can throw the weights far away and undo hours of training: the loss suddenly jumps or becomes `nan`.

**The idea.** Put a limit on the gradient's **length** (its norm, over all the network's parameters together). If it's longer than the limit, shrink it to the limit, keeping its direction; otherwise leave it alone.

$$
\lVert g \rVert = \sqrt{\textstyle\sum_{\text{all parameters}} g^2}, \qquad g \leftarrow g \cdot \min\left(1, \frac{c}{\lVert g \rVert}\right)
$$

🖼 **Picture.** Limit $c = 1$: the gradient $(3, 4)$ has length 5, so it's scaled by $1/5$ to $(0.6, 0.8)$, same direction. The short gradient (length 0.5) is unchanged.

In [ ]:
fig_clip()

**In PyTorch** it goes between `backward()` (which fills the gradients) and `step()` (which uses them). It returns the length before clipping, which is worth logging:

In [ ]:
w = torch.zeros(2, requires_grad=True)
(3 * w[0] + 4 * w[1]).backward()                                     # gradient (3, 4), length 5
total = torch.nn.utils.clip_grad_norm_([w], max_norm=1.0)            # between backward() and step()
print(total, w.grad)                                                 # → tensor(5.) tensor([0.6000, 0.8000])

**In one sentence:** gradient clipping rescales the whole gradient to a maximum length before the step, so one bad batch can't make a huge jump.

## 6. In Python: exercises from this notebook's block

The code cells of §1–5 together are this notebook's "In Python" block. Exercises 2–5 are its hand-written pieces: batch norm and layer norm by hand (checked against `nn.BatchNorm1d` and `nn.LayerNorm`), clipping (checked against `clip_grad_norm_`), and the residual block as a module.

**Exercise 2 · Batch norm, training mode.** $y = \gamma \frac{X - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta$ with $\mu$, $\sigma^2$ the mean and variance of **each column** over the batch (`axis=0`; `np.var` divides by $n$, as batch norm does). `X` `(n, d)`; `gamma`, `beta` `(d,)`. Return **shape `(n, d)`**.

**Example**: what goes in, and what should come out:
```python
batchnorm_train(X, np.ones(3), np.zeros(3))[:, 1].round(3)    # → [-0.447  1.342  0.447 -1.342]   column 1, as in §2
```

In [ ]:
def batchnorm_train(X, gamma, beta, eps=1e-5):
    pass  # your code

In [ ]:
check(2)

**Exercise 3 · Layer norm.** The same formula with the mean and variance of **each row** (`axis=1, keepdims=True`). `X` `(n, d)`; `gamma`, `beta` `(d,)`. Return **shape `(n, d)`**.

**Example**: what goes in, and what should come out:
```python
layernorm(X, np.ones(3), np.zeros(3))[2].round(3)    # → [-0.615  1.41  -0.795]   sample 2, as in §3
```

In [ ]:
def layernorm(X, gamma, beta, eps=1e-5):
    pass  # your code

In [ ]:
check(3)

**Exercise 4 · Clip by the global norm.** `grads` is a list of arrays (one per parameter). Compute the length of all of them together, $\sqrt{\sum g^2}$ over every entry of every array, and return a **new** list with every array multiplied by $\min(1, c / \text{length})$.

**Example**: what goes in, and what should come out:
```python
clip_by_global_norm([np.array([3.0]), np.array([4.0])], 1.0)    # → about [0.6] and [0.8]: length 5, scaled by 1/5
```

In [ ]:
def clip_by_global_norm(grads, max_norm):
    pass  # your code

In [ ]:
check(4)

**Exercise 5 · A residual block.** `ResidualBlock(d)`, a subclass of `nn.Module` with `self.fc1 = nn.Linear(d, d)` and `self.fc2 = nn.Linear(d, d)` (in that order). `forward(x)` returns $x + \text{fc2}(\text{ReLU}(\text{fc1}(x)))$.

**Example**: what goes in, and what should come out:
```python
blk = ResidualBlock(4)
blk(torch.zeros(2, 4)).shape    # → torch.Size([2, 4])   same shape in and out
```

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, d):
        pass  # your code

    def forward(self, x):
        pass  # your code

In [ ]:
check(5)

**Recipe (a deep network that trains).** He initialization for ReLU layers; `Linear → BatchNorm → ReLU` (or LayerNorm for tiny batches and text); residual connections around blocks once you go past about 10 layers; clip the gradient norm to about 1 if the loss ever spikes or turns `nan`; Adam or AdamW to optimize.

In [ ]:
for n in range(1, 6):
    check(n)

---
Solutions: `solutions/03_solutions.py`. Next: `04_convolution.ipynb`, networks for images.